# Rapport de Pipeline – Visualisations & Contrôles (v4, avec AVANT/APRÈS)

## Plan (mis à jour)
1. Setup alias de chemin + helpers d’affichage (robustes)
2. Charger la configuration TOML via alias virtuel (`rakuten://…`)
3. Charger les données (reconstruction de chemins + fallback + glob)
4. **Texte — AVANT vs APRÈS preprocessing** : top mots (unigrammes/bigrammes), longueurs, TF‑IDF (sublinear on/off)
5. **Images — AVANT vs APRÈS prétraitement** : stats (occupancy, white_ratio, black_ratio) et scatter
6. **Distribution des classes — AVANT vs APRÈS sampling adaptatif (simulé)** : caps par classe (Top 20)
7. **PCA/ACP images** : variance expliquée cumulée (échantillon)
8. Corrélations simples features↔classe (indicatif)
9. Mini matrice de confusion texte (fallback sans stratify)
10.Top termes TF‑IDF par classe via LR(OvR)


In [ ]:
# --- SETUP ALIAS DE CHEMIN + HELPERS PLOTS ---

import os
from pathlib import Path
import tomllib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

# Définir la racine virtuelle du projet
RAKUTEN_ROOT = os.environ.get("RAKUTEN_ROOT", "")
if RAKUTEN_ROOT:
    PROJECT_ROOT = Path(RAKUTEN_ROOT).resolve()
else:
    candidates = [Path.cwd()] + list(Path.cwd().parents)[:4]
    found = None
    for c in candidates:
        if (c / "features").exists() and (c / "data").exists():
            found = c
            break
    PROJECT_ROOT = found or Path.cwd()

print("[INFO] PROJECT_ROOT =", PROJECT_ROOT)

def vpath(uri_or_path: str) -> Path:
    """Convertir une URI 'rakuten://...' en chemin absolu sous PROJECT_ROOT, sinon normaliser en absolu."""
    p = str(uri_or_path)
    if p.startswith("rakuten://"):
        rel = p.replace("rakuten://", "")
        return (PROJECT_ROOT / rel).resolve()
    return Path(p).resolve()

def vopen(uri_or_path: str, mode="rb"):
    """Ouvrir un fichier en acceptant les URI 'rakuten://' ou un chemin direct."""
    return open(vpath(uri_or_path), mode)

# Dossier figures
FIG_DIR = (PROJECT_ROOT / "reports" / "figs")
FIG_DIR.mkdir(parents=True, exist_ok=True)
print("[INFO] FIG_DIR =", FIG_DIR)

# Helpers de tracé robustes
def safe_hist(series, title, xlabel, bins=50, fname=None):
    s = pd.Series(series).dropna()
    if s.size == 0:
        print(f"[SKIP] {title} : aucune donnée exploitable.")
        return
    plt.figure()
    plt.hist(s.values, bins=bins)
    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel("Fréquence")
    if fname:
        plt.tight_layout()
        plt.savefig(FIG_DIR / fname, dpi=120)
    plt.show()

def safe_bar_from_pairs(pairs, title, xlabel="Terme", ylabel="Fréquence", top=20, fname=None):
    pairs = [(k, v) for (k, v) in pairs if v and v > 0]
    if not pairs:
        print(f"[SKIP] {title} : aucune paire valide.")
        return
    pairs = pairs[:top]
    labels = [p[0] for p in pairs]
    vals   = [p[1] for p in pairs]
    x = np.arange(len(pairs))
    plt.figure(figsize=(max(6, len(pairs)*0.5), 4))
    plt.bar(x, vals)
    plt.xticks(x, labels, rotation=60, ha="right")
    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.tight_layout()
    if fname:
        plt.savefig(FIG_DIR / fname, dpi=120)
    plt.show()

def safe_show_confusion_matrix(cm, labels, title="Matrice de confusion (mini-échantillon)", fname=None):
    if cm is None or (hasattr(cm, "size") and cm.size == 0):
        print(f"[SKIP] {title} : matrice vide.")
        return
    plt.figure()
    plt.imshow(cm, aspect="auto")
    plt.title(title)
    plt.xlabel("Prédit")
    plt.ylabel("Vrai")
    plt.xticks(ticks=np.arange(len(labels)), labels=labels, rotation=45, ha="right")
    plt.yticks(ticks=np.arange(len(labels)), labels=labels)
    plt.colorbar()
    plt.tight_layout()
    if fname:
        plt.savefig(FIG_DIR / fname, dpi=120)
    plt.show()


In [ ]:
# --- CHARGER LA CONFIG (alias virtuel) ---
CFG_URI = os.environ.get("RAKUTEN_CFG", "rakuten://features/config.toml")
cfg_path = vpath(CFG_URI)

print("[INFO] CFG_URI      =", CFG_URI)
print("[INFO] cfg_path     =", cfg_path)
print("[INFO] cfg_path ok? =", cfg_path.exists())

with vopen(CFG_URI, "rb") as f:
    cfg = tomllib.load(f)

list(cfg.keys())


In [ ]:
# --- CHARGER LES CSV (reconstruction de chemins + fallback + glob) ---

def normalize_candidates(path_str, roots):
    out = []
    if isinstance(path_str, str) and path_str:
        p = Path(path_str)
        if p.is_absolute():
            out.append(p)
        else:
            out.extend([r / p for r in roots])
    return out

def find_first_by_glob(patterns, roots):
    for r in roots:
        for pat in patterns:
            matches = list(r.glob(pat))
            if matches:
                return matches[0]
    return None

def try_read_csv(candidates, fallback_patterns, label):
    for p in candidates:
        if p.exists():
            print(f"[OK] {label} chargé :", p)
            return pd.read_csv(p), str(p)
    found = find_first_by_glob(fallback_patterns, roots)
    if found and found.exists():
        print(f"[OK] {label} trouvé via recherche :", found)
        return pd.read_csv(found), str(found)
    print(f"[ERROR] {label} introuvable. Candidats testés :")
    for p in candidates:
        print("   -", p)
    print("     motifs glob:", fallback_patterns)
    raise FileNotFoundError(f"{label} non trouvé.")

# Racines à partir de la config et du notebook
roots = []
if cfg_path and cfg_path.exists() and len(cfg_path.parents) >= 2:
    roots.append(cfg_path.parents[1])
nb_roots = [Path.cwd()] + list(Path.cwd().parents)[:3]
for r in nb_roots:
    if r not in roots:
        roots.append(r)

print("[INFO] Racines candidates :")
for r in roots:
    print(" -", r)

# Chemins configurés
x_train_cfg = cfg.get('data', {}).get('x_train_csv', 'data/X_train_update.csv')
y_train_cfg = cfg.get('data', {}).get('y_train_csv', 'data/Y_train_CVw08PX.csv')
x_test_cfg  = cfg.get('data', {}).get('x_test_csv',  'data/X_test_update.csv')

# Candidats
x_train_candidates = normalize_candidates(x_train_cfg, roots) + [r/'data/X_train_update.csv' for r in roots] + [r/'data/X_train.csv' for r in roots]
y_train_candidates = normalize_candidates(y_train_cfg, roots) + [r/'data/Y_train_CVw08PX.csv' for r in roots] + [r/'data/Y_train.csv' for r in roots]
x_test_candidates  = normalize_candidates(x_test_cfg,  roots) + [r/'data/X_test_update.csv'  for r in roots] + [r/'data/X_test.csv'  for r in roots]

# Lecture
X_train, x_train_path = try_read_csv(x_train_candidates, ['**/X_train*.csv'], "X_train")
y_train, y_train_path = try_read_csv(y_train_candidates, ['**/Y_train*.csv'], "y_train")
X_test,  x_test_path  = try_read_csv(x_test_candidates,  ['**/X_test*.csv'],  "X_test")

# Harmoniser la cible
if 'prdtypecode' not in y_train.columns:
    if len(y_train.columns) == 1:
        y_train.columns = ['prdtypecode']
    else:
        for c in y_train.columns:
            if 'prd' in c.lower() or 'code' in c.lower():
                y_train.rename(columns={c:'prdtypecode'}, inplace=True)
                break
        if 'prdtypecode' not in y_train.columns:
            y_train.rename(columns={y_train.columns[0]:'prdtypecode'}, inplace=True)

print("[INFO] X_train utilisé :", x_train_path)
print("[INFO] y_train utilisé :", y_train_path)
print("[INFO] X_test  utilisé :", x_test_path)
display(X_train.head(3))
display(y_train['prdtypecode'].value_counts().head(10))


In [ ]:
# --- TEXTE : fonctions de preprocessing (stopwords/stemming/normalisation) ---

import re
import unicodedata

# Charger stopwords NLTK si dispo ; sinon fallback simple
try:
    import nltk
    try:
        from nltk.corpus import stopwords
        _ = stopwords.words('french')  # test
    except LookupError:
        nltk.download('stopwords')
    STOP_FR = set(stopwords.words('french'))
    STEMMER = None
    try:
        from nltk.stem.snowball import SnowballStemmer
        STEMMER = SnowballStemmer('french')
    except Exception:
        STEMMER = None
except Exception:
    print("[INFO] NLTK indisponible → fallback stopwords minimal.")
    STOP_FR = set(['le','la','les','de','des','du','un','une','et','à','a','au','aux','en','pour','par','avec','sur','dans','ce','cet','cette','ces','plus','moins','ou','où','se','sa','son','ses','leur','leurs','ne','pas','que','qui','quoi','dont'])
    STEMMER = None

def strip_accents(s: str) -> str:
    return ''.join(c for c in unicodedata.normalize('NFD', s) if unicodedata.category(c) != 'Mn')

def basic_normalize(text: str) -> str:
    # Mettre en minuscules, enlever accents, garder lettres/chiffres/espace
    t = strip_accents(text.lower())
    t = re.sub(r'[^a-z0-9\s]+', ' ', t)
    t = re.sub(r'\s+', ' ', t).strip()
    return t

def tokenize(text: str):
    return re.findall(r'[a-z0-9]+', text)

def clean_text(text: str, apply_stemming: bool = True) -> str:
    t = basic_normalize(text or '')
    tokens = [w for w in tokenize(t) if w not in STOP_FR and len(w) > 1]
    if apply_stemming and STEMMER is not None:
        tokens = [STEMMER.stem(w) for w in tokens]
    return ' '.join(tokens)


In [ ]:
# --- TEXTE : AVANT vs APRÈS preprocessing (top mots, longueurs) ---

# Colonnes de base
X_train['designation'] = X_train.get('designation', '').fillna('').astype(str)
X_train['description'] = X_train.get('description', '').fillna('').astype(str)

# Construire un texte concaténé
X_train['text_raw'] = (X_train['designation'] + ' ' + X_train['description']).str.strip()

# Créer version nettoyée (stopwords + stemming si dispo)
X_train['text_clean'] = X_train['text_raw'].apply(lambda s: clean_text(s, apply_stemming=True))

# Longueurs avant/après
X_train['raw_len']   = X_train['text_raw'].apply(len)
X_train['clean_len'] = X_train['text_clean'].apply(len)

# Histos AVANT / APRÈS (séparés)
safe_hist(X_train['raw_len'],   "Longueur des textes AVANT preprocessing",  "Nombre de caractères", bins=60, fname="text_len_raw_hist.png")
safe_hist(X_train['clean_len'], "Longueur des textes APRÈS preprocessing", "Nombre de caractères", bins=60, fname="text_len_clean_hist.png")

# Top unigrammes AVANT
from sklearn.feature_extraction.text import CountVectorizer
top_raw = []
raw_sample = X_train['text_raw'].head(4000).tolist()
try:
    cv_raw = CountVectorizer(ngram_range=(1,1), min_df=2)
    Xr = cv_raw.fit_transform(raw_sample)
    if Xr.shape[1] > 0:
        freq = np.asarray(Xr.sum(axis=0)).ravel()
        idx  = np.argsort(freq)[-20:][::-1]
        top_raw = [(cv_raw.get_feature_names_out()[i], int(freq[i])) for i in idx]
except Exception as e:
    print("[INFO] Unigrammes AVANT non calculables:", e)
safe_bar_from_pairs(top_raw, "Top unigrammes AVANT preprocessing", "Terme", "Fréquence", top=20, fname="text_top_unigrams_raw.png")

# Top unigrammes APRÈS
top_clean = []
clean_sample = X_train['text_clean'].head(4000).tolist()
try:
    cv_clean = CountVectorizer(ngram_range=(1,1), min_df=2)
    Xc = cv_clean.fit_transform(clean_sample)
    if Xc.shape[1] > 0:
        freq = np.asarray(Xc.sum(axis=0)).ravel()
        idx  = np.argsort(freq)[-20:][::-1]
        top_clean = [(cv_clean.get_feature_names_out()[i], int(freq[i])) for i in idx]
except Exception as e:
    print("[INFO] Unigrammes APRÈS non calculables:", e)
safe_bar_from_pairs(top_clean, "Top unigrammes APRÈS preprocessing", "Terme", "Fréquence", top=20, fname="text_top_unigrams_clean.png")

# Bigrams AVANT vs APRÈS
top_bi_raw = []
try:
    cv_bi_raw = CountVectorizer(ngram_range=(2,2), min_df=2)
    Xbr = cv_bi_raw.fit_transform(raw_sample)
    if Xbr.shape[1] > 0:
        freq = np.asarray(Xbr.sum(axis=0)).ravel()
        idx  = np.argsort(freq)[-15:][::-1]
        top_bi_raw = [(cv_bi_raw.get_feature_names_out()[i], int(freq[i])) for i in idx]
except Exception as e:
    print("[INFO] Bigrams AVANT non calculables:", e)
safe_bar_from_pairs(top_bi_raw, "Top bigrammes AVANT preprocessing", "Terme", "Fréquence", top=15, fname="text_top_bigrams_raw.png")

top_bi_clean = []
try:
    cv_bi_clean = CountVectorizer(ngram_range=(2,2), min_df=2)
    Xbc = cv_bi_clean.fit_transform(clean_sample)
    if Xbc.shape[1] > 0:
        freq = np.asarray(Xbc.sum(axis=0)).ravel()
        idx  = np.argsort(freq)[-15:][::-1]
        top_bi_clean = [(cv_bi_clean.get_feature_names_out()[i], int(freq[i])) for i in idx]
except Exception as e:
    print("[INFO] Bigrams APRÈS non calculables:", e)
safe_bar_from_pairs(top_bi_clean, "Top bigrammes APRÈS preprocessing", "Terme", "Fréquence", top=15, fname="text_top_bigrams_clean.png")


In [ ]:
# --- TEXTE : comparaison TF-IDF sublinear_tf False vs True (sur textes nettoyés) ---

from sklearn.feature_extraction.text import TfidfVectorizer

mini = X_train['text_clean'].head(5000).tolist()
# TF-IDF sans sublinear
vec_no = TfidfVectorizer(max_features=50000, ngram_range=(1,2), sublinear_tf=False)
try:
    X_no = vec_no.fit_transform(mini)
    # Norme L2 des documents
    norms_no = np.sqrt((X_no.multiply(X_no)).sum(axis=1)).A.ravel()
    safe_hist(norms_no, "Norme L2 des vecteurs TF‑IDF (sublinear_tf=False)", "Norme L2", bins=40, fname="tfidf_norms_no_sublinear.png")
except Exception as e:
    print("[INFO] TF-IDF (False) non calculable:", e)

# TF-IDF avec sublinear
vec_yes = TfidfVectorizer(max_features=50000, ngram_range=(1,2), sublinear_tf=True)
try:
    X_yes = vec_yes.fit_transform(mini)
    norms_yes = np.sqrt((X_yes.multiply(X_yes)).sum(axis=1)).A.ravel()
    safe_hist(norms_yes, "Norme L2 des vecteurs TF‑IDF (sublinear_tf=True)", "Norme L2", bins=40, fname="tfidf_norms_yes_sublinear.png")
except Exception as e:
    print("[INFO] TF-IDF (True) non calculable:", e)


In [ ]:
# --- IMAGES : AVANT vs APRÈS prétraitement ---

from PIL import Image, ImageOps, ImageFilter

img_train_dir = vpath(cfg.get("images", {}).get("train_dir", "rakuten://data/images/images/image_train"))
white_th = int(cfg.get("images", {}).get("stats", {}).get("white_threshold", 230))
black_th = int(cfg.get("images", {}).get("stats", {}).get("black_threshold", 25))

def find_image_path(row):
    imgid = row.get("imageid")
    pid   = row.get("productid")
    if pd.isna(imgid) or pd.isna(pid):
        return None
    return img_train_dir / f"image_{int(imgid)}_product_{int(pid)}.jpg"

def compute_stats_from_array(arr, white_th=230, black_th=25):
    H, W = arr.shape
    total = max(1, H*W)
    white_ratio = float((arr >= white_th).sum()) / total
    black_ratio = float((arr <= black_th).sum()) / total
    obj_mask = (arr > black_th) & (arr < white_th)
    occupancy = float(obj_mask.sum()) / total
    return occupancy, white_ratio, black_ratio

def preprocess_pil(im):
    # Convertir en niveaux de gris, auto-contrast, léger lissage, redimensionner
    g = im.convert("L")
    g = ImageOps.autocontrast(g)
    g = g.filter(ImageFilter.MedianFilter(size=3))
    g = g.resize((256,256))
    return g

if not img_train_dir.exists():
    print("[WARN] Dossier images introuvable :", img_train_dir)
else:
    X_tmp = X_train.head(200).copy()
    if "imageid" in X_tmp.columns and "productid" in X_tmp.columns:
        X_tmp["img_path"] = X_tmp.apply(find_image_path, axis=1)
        X_tmp = X_tmp[X_tmp["img_path"].apply(lambda p: p is not None and Path(p).exists())].head(60)
        if X_tmp.empty:
            print("[WARN] Aucune image trouvée dans", img_train_dir)
        else:
            # Calculer stats AVANT/APRÈS
            before_occ, before_w, before_b = [], [], []
            after_occ,  after_w,  after_b  = [], [], []
            for p in X_tmp["img_path"].tolist():
                try:
                    im = Image.open(p)
                    arr_before = np.array(im.convert("L"), dtype=np.uint8)
                    occ_b, w_b, b_b = compute_stats_from_array(arr_before, white_th, black_th)
                    before_occ.append(occ_b); before_w.append(w_b); before_b.append(b_b)

                    im2 = preprocess_pil(im)
                    arr_after = np.array(im2, dtype=np.uint8)
                    occ_a, w_a, b_a = compute_stats_from_array(arr_after, white_th, black_th)
                    after_occ.append(occ_a); after_w.append(w_a); after_b.append(b_a)
                except Exception:
                    continue

            safe_hist(before_occ, "Occupancy AVANT prétraitement", "occupancy", bins=20, fname="img_occ_before.png")
            safe_hist(after_occ,  "Occupancy APRÈS prétraitement", "occupancy", bins=20, fname="img_occ_after.png")
            safe_hist(before_w,   "White_ratio AVANT prétraitement", "white_ratio", bins=20, fname="img_white_before.png")
            safe_hist(after_w,    "White_ratio APRÈS prétraitement", "white_ratio", bins=20, fname="img_white_after.png")
            safe_hist(before_b,   "Black_ratio AVANT prétraitement", "black_ratio", bins=20, fname="img_black_before.png")
            safe_hist(after_b,    "Black_ratio APRÈS prétraitement", "black_ratio", bins=20, fname="img_black_after.png")

            # Scatter APRÈS prétraitement
            if len(after_occ) > 0 and len(after_w) > 0:
                plt.figure()
                plt.scatter(after_occ, after_w, s=10)
                plt.title("Scatter occupancy vs white_ratio APRÈS prétraitement")
                plt.xlabel("occupancy")
                plt.ylabel("white_ratio")
                plt.tight_layout()
                plt.savefig(FIG_DIR / "img_scatter_after.png", dpi=120)
                plt.show()
    else:
        print("[WARN] Colonnes imageid/productid manquantes dans X_train.")


In [ ]:
# --- DISTRIBUTION DES CLASSES : AVANT vs APRÈS sampling adaptatif (simulation) ---

# Construire distribution brute
if 'prdtypecode' in y_train.columns:
    vc = y_train['prdtypecode'].value_counts()
    pairs_before = [(str(int(k)), int(v)) for k, v in vc.head(20).items()]
    safe_bar_from_pairs(pairs_before, "Distribution des classes AVANT sampling (Top 20)", xlabel="Classe", ylabel="Effectif", top=20, fname="cls_dist_before.png")

    # Définir un cap : utiliser soit cfg['sampling']['under_cap_dict'] si présent, soit un cap global (ex: 5000)
    cap_dict = {}
    try:
        cap_dict = cfg.get('sampling', {}).get('under_cap_dict', {}) or {}
    except Exception:
        cap_dict = {}
    if not cap_dict:
        # Cap global = quantile 80% des effectifs (ex) ou 5000 par défaut
        cap_global = int(np.percentile(vc.values, 80)) if vc.size > 0 else 5000
        cap_global = max(1000, cap_global)
        cap_dict = {int(k): int(min(v, cap_global)) for k, v in vc.items()}

    # Simuler l'under-sampling par cap
    after_counts = {}
    for cls, n in vc.items():
        cap = cap_dict.get(int(cls), int(n))
        after_counts[int(cls)] = int(min(n, cap))

    # Afficher distribution APRÈS
    after_sorted = sorted(after_counts.items(), key=lambda kv: kv[1], reverse=True)[:20]
    pairs_after = [(str(k), v) for k, v in after_sorted]
    safe_bar_from_pairs(pairs_after, "Distribution des classes APRÈS sampling (Top 20 simulé)", xlabel="Classe", ylabel="Effectif (cap)", top=20, fname="cls_dist_after.png")
else:
    print("[SKIP] prdtypecode absent : pas de distribution de classes.")


In [ ]:
# --- PCA/ACP images : variance expliquée cumulée (échantillon) ---

from sklearn.decomposition import PCA
from PIL import Image

def load_flatten_images(paths, max_n=200, size=(64,64)):
    arrs = []
    for i, p in enumerate(list(paths)[:max_n]):
        try:
            im = Image.open(p).convert("L").resize(size)
            ar = np.array(im, dtype=np.float32).reshape(-1)
            arrs.append(ar)
        except Exception:
            continue
    if not arrs:
        return None
    return np.stack(arrs, axis=0)

if 'X_tmp' in globals() and isinstance(X_tmp, pd.DataFrame) and 'img_path' in X_tmp.columns and not X_tmp.empty:
    A = load_flatten_images(X_tmp['img_path'], max_n=200, size=(64,64))
    if A is not None and A.shape[0] >= 5:
        pca = PCA(n_components=min(100, A.shape[0]-1), random_state=42)
        pca.fit(A)
        evr = pca.explained_variance_ratio_
        plt.figure()
        plt.plot(np.arange(1, len(evr)+1), np.cumsum(evr))
        plt.title("PCA images — variance expliquée cumulée")
        plt.xlabel("Nombre de composantes")
        plt.ylabel("Variance expliquée cumulée")
        plt.tight_layout()
        plt.savefig(FIG_DIR / "img_pca_cumsum.png", dpi=120)
        plt.show()
    else:
        print("[SKIP] PCA images : pas assez d'images exploitables.")
else:
    print("[SKIP] PCA images : échantillon d'images indisponible.")


In [ ]:
# --- Corrélations simples features ↔ classe (indicatif) ---

def corr_std(a, b):
    a = pd.Series(a).astype(float)
    b = pd.Series(b).astype(float)
    a = (a - a.mean()) / (a.std() + 1e-9)
    b = (b - b.mean()) / (b.std() + 1e-9)
    return float((a * b).mean())

if 'prdtypecode' in y_train.columns:
    # Tenter d'utiliser les features images après prétraitement si dispo (after_occ/after_w, etc.)
    if 'X_tmp' in globals() and isinstance(X_tmp, pd.DataFrame) and 'img_path' in X_tmp.columns and not X_tmp.empty:
        # recompute quickly if vectors exist
        base = pd.DataFrame({
            'prdtypecode': y_train['prdtypecode'].head(len(X_tmp)).values
        })
        # fallback: utiliser stats AVANT si colonnes pas en mémoire
        try:
            base['occupancy'] = pd.Series(after_occ[:len(base)]) if 'after_occ' in globals() else np.nan
            base['white_ratio'] = pd.Series(after_w[:len(base)]) if 'after_w' in globals() else np.nan
            base['black_ratio'] = pd.Series(after_b[:len(base)]) if 'after_b' in globals() else np.nan
        except Exception:
            pass
    else:
        base = X_train.head(300).join(y_train, how='left')

    out = {}
    for col in ['occupancy','white_ratio','black_ratio']:
        if col in base.columns:
            out[f'corr_{col}_vs_class'] = corr_std(base[col].fillna(base[col].mean()), base['prdtypecode'].astype('category').cat.codes)
    out if out else print("[SKIP] Pas de features images pour corrélation.")
else:
    print("[SKIP] prdtypecode absent.")


In [ ]:
# --- (Option) Mini matrice de confusion texte ---

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix

mini = X_train['text_clean'].fillna('').astype(str).head(4000)
if not mini.empty and 'prdtypecode' in y_train.columns:
    mini_y = y_train['prdtypecode'].head(len(mini)).astype(int).values
    vec = TfidfVectorizer(max_features=50000, ngram_range=(1,2), sublinear_tf=True)
    try:
        Xv = vec.fit_transform(mini.tolist())
        try:
            X_tr, X_te, y_tr, y_te = train_test_split(Xv, mini_y, test_size=0.25, random_state=42, stratify=mini_y)
        except ValueError as e:
            print("[INFO] Stratify impossible :", e, "→ fallback sans stratify")
            X_tr, X_te, y_tr, y_te = train_test_split(Xv, mini_y, test_size=0.25, random_state=42, stratify=None)

        clf = LogisticRegression(max_iter=2000, solver="saga")
        clf.fit(X_tr, y_tr)
        y_pr = clf.predict(X_te)

        top_classes = pd.Series(y_te).value_counts().head(10).index.tolist()
        mask = np.isin(y_te, top_classes)
        cm = confusion_matrix(y_te[mask], y_pr[mask], labels=top_classes)
        safe_show_confusion_matrix(cm, labels=top_classes, title="Matrice de confusion (Top 10 classes)", fname="cm_top10.png")
    except ValueError as e:
        print("[INFO] Vectorisation / fit non possible :", e)
else:
    print("[SKIP] Données insuffisantes pour la matrice de confusion.")


In [ ]:
# --- (Option) Top termes TF‑IDF par classe via LR (OvR) ---

from sklearn.multiclass import OneVsRestClassifier
from sklearn.feature_extraction.text import TfidfVectorizer

mini = X_train['text_clean'].fillna('').astype(str).head(6000)
if not mini.empty and 'prdtypecode' in y_train.columns:
    mini_y = y_train['prdtypecode'].head(len(mini)).astype(int).values
    vec = TfidfVectorizer(max_features=80000, ngram_range=(1,2), sublinear_tf=True)
    Xv = vec.fit_transform(mini.tolist())

    clf = OneVsRestClassifier(LogisticRegression(max_iter=2000, solver="saga"))
    clf.fit(Xv, mini_y)

    feature_names = vec.get_feature_names_out()
    class_labels = np.unique(mini_y)

    top3 = pd.Series(mini_y).value_counts().head(3).index.tolist()
    tops = {}
    for cls in top3:
        est = clf.estimators_[list(class_labels).index(cls)]
        coefs = est.coef_.ravel()
        top_idx = np.argsort(coefs)[-15:][::-1]
        tops[int(cls)] = [feature_names[i] for i in top_idx]
    tops
else:
    print("[SKIP] Pas assez de données pour extraire des top termes.")
